# ATLAS tutorial: download GLOFAS river discharge data from Copernicus EWDS

This notebook downloads historical river discharge data from **GLOFAS** using the Copernicus **Early Warning Data Store (EWDS)** API.

The notebook is organised as a step-by-step tutorial for users who are not Python experts. In most cases, users only need to edit the **Input parameters** section and then run the notebook from top to bottom.

## What this notebook does

1. Imports the required Python packages.
2. Defines the input parameters: API access, country/area, dates and output folder.
3. Defines helper functions used for validation and download.
4. Downloads GLOFAS river discharge month by month.

Documentation: https://global-flood.emergency.copernicus.eu/react/

## Before running

You need access to the Copernicus EWDS/CDS API and a personal API key. Do **not** share your personal API key publicly.

Recommended option: configure your key in the standard local configuration file on your machine. If that is already configured, you can leave `EWDS_API_KEY = None` below.

Alternative option: paste your personal key in `EWDS_API_KEY`, but only in your private local copy of this notebook.

## Output

The notebook saves one file per month. By default, files are saved in a local folder called `data/glofas_downloads/<country>/` inside the folder where this notebook is located.

## Step 1. Import required packages

Run this cell first. If `cdsapi` is not installed, install it with:

```bash
pip install cdsapi
```

In [2]:
from pathlib import Path
from datetime import datetime, timedelta

import cdsapi

## Step 2. Input parameters

Edit only this section for normal use.

### API access

- Create an account on Copernicus and you will access to your personal key on the User Profile Page
- `EWDS_API_KEY`: your personal API key. Leave it as `None` if your key is already configured locally.
- `EWDS_API_URL`: API endpoint used for GLOFAS downloads. Usually this should not be changed.

### Dataset selection

- `DATASET_NAME`: Copernicus dataset used for the download.
- `SYSTEM_VERSION`, `HYDROLOGICAL_MODEL`, `PRODUCT_TYPE`, `VARIABLE`: GLOFAS request parameters.
- `DATA_FORMAT`: output format requested from the API. The default is `grib2`, following the original notebook.

### Country and area

- `COUNTRY`: used to select a predefined bounding box.
- `AREAS`: dictionary of country bounding boxes in Copernicus format: `[North, West, South, East]`.

To add a new country, add a new entry to `AREAS` and set `COUNTRY` to the same name.

### Dates

- `START_DATE`: first date to download.
- `END_DATE`: last date to download.

The data are downloaded month by month. Each output file contains one month.

### Output folder

- `OUTPUT_DIR`: base folder where files will be saved. By default, files are saved in a local `data/glofas_downloads` folder.

Avoid absolute machine-specific paths such as `/mnt/DATA/...` when sharing the notebook with other users.

In [3]:
# =========================
# EWDS / CDS API ACCESS
# =========================

# Leave as None if your API key is already configured locally.
EWDS_API_KEY = ''

# Paste your key here only in your private local copy, for example:
# EWDS_API_KEY = "your-user-id:your-api-key"
# or, depending on your account configuration:
# EWDS_API_KEY = "your-api-key"

EWDS_API_URL = "https://ewds.climate.copernicus.eu/api"


# =========================
# DATASET SELECTION
# =========================

DATASET_NAME = "cems-glofas-historical"
SYSTEM_VERSION = "version_4_0"
HYDROLOGICAL_MODEL = "lisflood"
PRODUCT_TYPE = "consolidated"
VARIABLE = "river_discharge_in_the_last_24_hours"
DATA_FORMAT = "grib2"
DOWNLOAD_FORMAT = "unarchived"


# =========================
# COUNTRY / AREA SELECTION
# =========================

# Country name must match one key in the AREAS dictionary below.
COUNTRY = "ecuador"

# Bounding boxes in Copernicus format: [North, West, South, East]
AREAS = {
    "bolivia": [-9, -70, -24, -57],
    "ecuador": [1.9, -92.0, -5.3, -75.1],
}
AREA = AREAS[COUNTRY]


# =========================
# DATES
# =========================

START_DATE = datetime(1996, 1, 1)
END_DATE = datetime(2026, 12, 31)


# =========================
# OUTPUT FOLDER
# =========================

# Generic path, portable across computers.
# Files will be saved in: data/glofas_downloads/<COUNTRY>/
OUTPUT_DIR = Path("../data") / "glofas_downloads"

## Step 3. Functions

Run this cell without editing it.

The functions below:

- validate the main user inputs before starting the download;
- create the API client;
- create a clean output folder for the selected country;
- download GLOFAS river discharge month by month;
- skip files that already exist, unless overwrite is enabled.

In [4]:
def validate_inputs(country, area, start_date, end_date, output_dir, data_format):
    """
    Validate the main notebook inputs before starting the download.
    """
    if not isinstance(country, str) or not country.strip():
        raise ValueError("COUNTRY must be a non-empty string.")

    if not isinstance(area, list) or len(area) != 4:
        raise ValueError("AREA must be a list with four values: [North, West, South, East].")

    if start_date > end_date:
        raise ValueError("START_DATE must be earlier than or equal to END_DATE.")

    if data_format not in ["grib2", "netcdf"]:
        raise ValueError("DATA_FORMAT should usually be 'grib2' or 'netcdf'.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)


def create_ewds_client(api_key=None, api_url="https://ewds.climate.copernicus.eu/api"):
    """
    Create an EWDS/CDS API client.

    If api_key is None, cdsapi will use the standard local configuration file.
    """
    if api_key is None:
        return cdsapi.Client(url=api_url, progress=False)

    return cdsapi.Client(url=api_url, key=str(api_key), progress=False)


def get_output_extension(data_format):
    """
    Return the file extension to use for each downloaded file.
    """
    if data_format == "grib2":
        return "grib"
    if data_format == "netcdf":
        return "nc"
    return data_format


def get_country_output_dir(output_dir, country):
    """
    Create and return the output folder for the selected country.
    """
    country_output_dir = Path(output_dir) / country.lower()
    country_output_dir.mkdir(parents=True, exist_ok=True)
    return country_output_dir


def download_glofas_river_discharge_by_month(
    start_date,
    end_date,
    output_dir,
    country,
    area=None,
    cds_client=None,
    dataset_name="cems-glofas-historical",
    system_version="version_4_0",
    hydrological_model="lisflood",
    product_type="consolidated",
    variable="river_discharge_in_the_last_24_hours",
    data_format="grib2",
    download_format="unarchived",
    overwrite=False,
):
    """
    Download GLOFAS river discharge data one month at a time.

    Each month is saved as a separate file in:
    output_dir / country

    Parameters
    ----------
    start_date : datetime
        Download start date.
    end_date : datetime
        Download end date.
    output_dir : str or pathlib.Path
        Base output directory where files will be saved.
    country : str
        Country or region name used in the output folder and file names.
    area : list or None
        Geographic bounding box in Copernicus format: [North, West, South, East].
    cds_client : cdsapi.Client or None
        Existing API client. If None, a new client is created.
    overwrite : bool
        If False, existing files are skipped. If True, existing files are downloaded again.
    """
    if cds_client is None:
        cds_client = create_ewds_client()

    country_output_dir = get_country_output_dir(output_dir, country)
    output_extension = get_output_extension(data_format)

    current_date = start_date.replace(day=1)

    while current_date <= end_date:
        year = current_date.year
        month = current_date.strftime("%m")

        output_file = country_output_dir / f"glofas_{variable}_{country.lower()}_{year}_{month}.{output_extension}"

        if output_file.exists() and not overwrite:
            print(f"Skipping existing file: {output_file}")
            current_date += timedelta(days=31)
            current_date = current_date.replace(day=1)
            continue

        request_params = {
            "system_version": [system_version],
            "hydrological_model": [hydrological_model],
            "product_type": [product_type],
            "variable": [variable],
            "hyear": str(year),
            "hmonth": month,
            "hday": [f"{day:02d}" for day in range(1, 32)],
            "data_format": data_format,
            "download_format": download_format,
        }

        if area is not None:
            request_params["area"] = area

        print(f"Downloading GLOFAS river discharge | {country} | {year}-{month} ...")
        cds_client.retrieve(dataset_name, request_params, str(output_file))
        print(f"Saved to: {output_file}")

        current_date += timedelta(days=31)
        current_date = current_date.replace(day=1)

## Step 4. Check the selected configuration

Run this cell before starting the download. It prints the main settings so that users can quickly check whether the selected country, dates and output folder are correct.

In [5]:
validate_inputs(
    country=COUNTRY,
    area=AREA,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    data_format=DATA_FORMAT,
)

print("Configuration OK")
print(f"Country: {COUNTRY}")
print(f"Area [North, West, South, East]: {AREA}")
print(f"Period: {START_DATE:%Y-%m-%d} to {END_DATE:%Y-%m-%d}")
print(f"Output folder: {get_country_output_dir(OUTPUT_DIR, COUNTRY)}")
print(f"Dataset: {DATASET_NAME}")
print(f"Variable: {VARIABLE}")

Configuration OK
Country: ecuador
Area [North, West, South, East]: [1.9, -92.0, -5.3, -75.1]
Period: 1991-01-01 to 1991-01-31
Output folder: ../data/glofas_downloads/ecuador
Dataset: cems-glofas-historical
Variable: river_discharge_in_the_last_24_hours


## Step 5. Run the download

Run this cell after checking the input parameters above.

Set `OVERWRITE_EXISTING_FILES = True` only if you want to download files again even when they already exist in the output folder.

In [6]:
OVERWRITE_EXISTING_FILES = False

cds_client = create_ewds_client(
    api_key=EWDS_API_KEY,
    api_url=EWDS_API_URL,
)

download_glofas_river_discharge_by_month(
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    country=COUNTRY,
    area=AREA,
    cds_client=cds_client,
    dataset_name=DATASET_NAME,
    system_version=SYSTEM_VERSION,
    hydrological_model=HYDROLOGICAL_MODEL,
    product_type=PRODUCT_TYPE,
    variable=VARIABLE,
    data_format=DATA_FORMAT,
    download_format=DOWNLOAD_FORMAT,
    overwrite=OVERWRITE_EXISTING_FILES,
)

2026-06-12 10:01:02,609 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).
2026-06-12 10:01:02,772 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-06-12 10:01:02,774 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-06-12 10:01:02,775 INFO Request ID is 906735fd-240c-42d2-bb9c-ab7940d7ceb6


2026-06-12 10:01:04,848 INFO status has been updated to accepted
2026-06-12 10:01:17,333 INFO status has been updated to running
2026-06-12 10:01:41,605 INFO status has been updated to successful


Saved to: ../data/glofas_downloads/ecuador/glofas_river_discharge_in_the_last_24_hours_ecuador_1991_01.grib


## Notes for adapting this notebook to a new country or region

To use this notebook for a new country or region:

1. Add a new bounding box to the `AREAS` dictionary.
2. Set `COUNTRY` to the new dictionary key.
3. Check `START_DATE` and `END_DATE`.
4. Check `OUTPUT_DIR`.
5. Run all cells from top to bottom.

The bounding box must follow the Copernicus order: `[North, West, South, East]`.

## Notes for less experienced Python users

For normal use, you should only edit **Step 2. Input parameters**. The function cells can be run without changes.

If a download fails, check first that:

- the API key is correctly configured;
- the selected dataset is available for your account;
- the selected country name matches exactly one key in `AREAS`;
- the output folder is writable on your computer.